In [ ]:
#En este notebook voy a revisar con más detalle Posicion_cierre, porque la regresión logística obtuvo un resultado demasiado alto y vi que casi 
# todo depende de esta variable. Voy a comprobar qué pasa al quitarla, cuánto consigue por sí sola y si su comportamiento se mantiene a lo largo
# de los años. Con esto quiero asegurarme de que el resultado sea real y no se deba a alguna particularidad de los datos.

from pathlib import Path
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Localizo la carpeta principal del proyecto
ruta_proyecto = Path.cwd()

if ruta_proyecto.name == "notebooks":
    ruta_proyecto = ruta_proyecto.parent

archivo_particiones = (
    ruta_proyecto
    / "data"
    / "processed"
    / "eurusd_yahoo_particiones.csv"
)

datos = pd.read_csv(
    archivo_particiones,
    parse_dates=["Date"],
    index_col="Date"
)

datos["Objetivo"] = datos["Objetivo"].astype("Int64")

entrenamiento = datos[
    datos["Particion"] == "entrenamiento"
].copy()

validacion = datos[
    datos["Particion"] == "validacion"
].copy()

y_entrenamiento = entrenamiento["Objetivo"].astype(int)
y_validacion = validacion["Objetivo"].astype(int)

In [3]:
variables_predictoras = [
    "Retorno_diario",
    "Retorno_lag_1",
    "Retorno_lag_2",
    "Retorno_lag_3",
    "Retorno_lag_5",
    "Rango_diario",
    "Cuerpo_vela",
    "Posicion_cierre",
    "Distancia_MA5",
    "Distancia_MA10",
    "Distancia_MA20",
    "Volatilidad_5",
    "Volatilidad_20",
    "RSI_14",
    "MACD_hist"
]

X_entrenamiento = entrenamiento[variables_predictoras]
X_validacion = validacion[variables_predictoras]

In [4]:
def calcular_metricas(
    nombre, # del modelo
    y_real,
    y_predicho,
    probabilidades=None 
):
    resultado = {
        "Modelo": nombre,
        "Accuracy": accuracy_score(
            y_real,
            y_predicho
        ),
        "Balanced_accuracy": balanced_accuracy_score(
            y_real,
            y_predicho
        ),
        "Precision": precision_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "Recall": recall_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "F1": f1_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "ROC_AUC": None
    }

    if probabilidades is not None:
        resultado["ROC_AUC"] = roc_auc_score(
            y_real,
            probabilidades
        )

    return resultado

In [5]:
modelo_logistico = Pipeline([ 
    (
        "escalador",
        StandardScaler()
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

modelo_logistico.fit(
    X_entrenamiento,
    y_entrenamiento
)



pred_logistica = modelo_logistico.predict(
    X_validacion
)

prob_logistica = modelo_logistico.predict_proba(
    X_validacion
)[:, 1]


In [6]:
print(
    "Entrenamiento:",
    X_entrenamiento.index.min(),
    "hasta",
    X_entrenamiento.index.max()
)

print(
    "Validación:",
    X_validacion.index.min(),
    "hasta",
    X_validacion.index.max()
)

print(
    "¿Objetivo está dentro de X?:",
    "Objetivo" in X_entrenamiento.columns
)

print("Variables utilizadas:")
print(X_entrenamiento.columns.tolist())

Entrenamiento: 2004-01-29 00:00:00 hasta 2022-12-30 00:00:00
Validación: 2023-01-02 00:00:00 hasta 2024-12-31 00:00:00
¿Objetivo está dentro de X?: False
Variables utilizadas:
['Retorno_diario', 'Retorno_lag_1', 'Retorno_lag_2', 'Retorno_lag_3', 'Retorno_lag_5', 'Rango_diario', 'Cuerpo_vela', 'Posicion_cierre', 'Distancia_MA5', 'Distancia_MA10', 'Distancia_MA20', 'Volatilidad_5', 'Volatilidad_20', 'RSI_14', 'MACD_hist']


In [7]:
comprobacion = validacion[["Close", "Objetivo"]].copy()

comprobacion["Cierre_siguiente"] = (
    datos["Close"]
    .shift(-1)
    .reindex(comprobacion.index)
)

comprobacion["Objetivo_recalculado"] = (
    comprobacion["Cierre_siguiente"]
    > comprobacion["Close"]
).astype(int)

print(
    "Objetivos correctamente alineados:",
    (
        comprobacion["Objetivo"]
        == comprobacion["Objetivo_recalculado"]
    ).mean()
)

display(comprobacion.head())

Objetivos correctamente alineados: 1.0


,Close,Objetivo,Cierre_siguiente,Objetivo_recalculado
Date,,,,
2023-01-02,1.070973,0,1.067771,0
2023-01-03,1.067771,0,1.054685,0
2023-01-04,1.054685,1,1.060637,1
2023-01-05,1.060637,0,1.052222,0
2023-01-06,1.052222,1,1.065632,1


In [8]:
correlaciones = pd.concat(
    [
        X_entrenamiento,
        y_entrenamiento.rename("Objetivo")
    ],
    axis=1
).corr()["Objetivo"].drop("Objetivo")

correlaciones = correlaciones.sort_values(
    key=abs,
    ascending=False
)

display(correlaciones)

Posicion_cierre   -0.463882
Distancia_MA5     -0.058335
Distancia_MA10    -0.057539
Retorno_diario    -0.045235
Distancia_MA20    -0.039522
MACD_hist         -0.038416
Cuerpo_vela       -0.025711
Retorno_lag_5     -0.025599
RSI_14            -0.023192
Retorno_lag_2     -0.020666
Retorno_lag_1     -0.012069
Rango_diario      -0.012054
Retorno_lag_3     -0.010914
Volatilidad_20    -0.001447
Volatilidad_5      0.000211
Name: Objetivo, dtype: float64

In [9]:
variables_sin_posicion = [
    variable
    for variable in variables_predictoras
    if variable != "Posicion_cierre"
]

X_entrenamiento_sin_posicion = entrenamiento[
    variables_sin_posicion
]

X_validacion_sin_posicion = validacion[
    variables_sin_posicion
]

modelo_sin_posicion = Pipeline([
    ("escalador", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=1000))
])

modelo_sin_posicion.fit(
    X_entrenamiento_sin_posicion,
    y_entrenamiento
)

pred_sin_posicion = modelo_sin_posicion.predict(
    X_validacion_sin_posicion
)

prob_sin_posicion = modelo_sin_posicion.predict_proba(
    X_validacion_sin_posicion
)[:, 1]

calcular_metricas(
    "Regresión logística sin Posicion_cierre",
    y_validacion,
    pred_sin_posicion,
    prob_sin_posicion
)

{'Modelo': 'Regresión logística sin Posicion_cierre',
 'Accuracy': 0.4731800766283525,
 'Balanced_accuracy': 0.473015873015873,
 'Precision': 0.4555984555984556,
 'Recall': 0.46825396825396826,
 'F1': 0.461839530332681,
 'ROC_AUC': 0.4907113462669018}

In [10]:
comparacion_posicion = validacion[
    ["Posicion_cierre", "Objetivo"]
].copy()

comparacion_posicion["Prediccion_simple"] = (
    comparacion_posicion["Posicion_cierre"] < 0.5
).astype(int)

calcular_metricas(
    "Regla simple Posicion_cierre",
    y_validacion,
    comparacion_posicion["Prediccion_simple"]
)

{'Modelo': 'Regla simple Posicion_cierre',
 'Accuracy': 0.7969348659003831,
 'Balanced_accuracy': 0.7970899470899471,
 'Precision': 0.7829457364341085,
 'Recall': 0.8015873015873016,
 'F1': 0.792156862745098,
 'ROC_AUC': None}

In [11]:
pred_posicion_entrenamiento = (
    entrenamiento["Posicion_cierre"] < 0.5
).astype(int)

calcular_metricas(
    "Regla Posicion_cierre en entrenamiento",
    y_entrenamiento,
    pred_posicion_entrenamiento
)

{'Modelo': 'Regla Posicion_cierre en entrenamiento',
 'Accuracy': 0.7143439282803586,
 'Balanced_accuracy': 0.7143102193980904,
 'Precision': 0.7183451734224823,
 'Recall': 0.7024928483857785,
 'F1': 0.7103305785123967,
 'ROC_AUC': None}

In [12]:
analisis_estabilidad = pd.concat([
    entrenamiento,
    validacion
]).copy()

analisis_estabilidad["Prediccion_simple"] = (
    analisis_estabilidad["Posicion_cierre"] < 0.5
).astype(int)

analisis_estabilidad["Acierto"] = (
    analisis_estabilidad["Prediccion_simple"]
    == analisis_estabilidad["Objetivo"]
).astype(int)

resultado_anual = (
    analisis_estabilidad
    .groupby(analisis_estabilidad.index.year)["Acierto"]
    .agg(["mean", "count"])
    .rename(columns={
        "mean": "Accuracy",
        "count": "Jornadas"
    })
    .round(4)
)

display(resultado_anual)

,Accuracy,Jornadas
Date,,
2004,0.5643,241
2005,0.5290,259
2006,0.5115,260
2007,0.5155,258
2008,0.5391,243
2009,0.5402,261
2010,0.6169,261
2011,0.7962,260
2012,0.7769,260


# Comparación con Alpha Vantage

Para comprobar si el comportamiento de Posicion_cierre también aparece en otra fuente, voy a descargar los datos diarios del EUR/USD desde Alpha Vantage. La intención no es sustituir todavía el dataset principal, sino revisar si la relación observada con Yahoo Finance se repite utilizando otra estructura OHLC.

In [13]:
import os
import requests

from dotenv import load_dotenv

# Cargo la clave guardada en el archivo .env
load_dotenv(ruta_proyecto / ".env")

api_key = os.getenv("ALPHA_VANTAGE_API_KEY")

if not api_key:
    raise ValueError(
        "No se encontró la clave de Alpha Vantage en el archivo .env"
    )

print("Clave cargada correctamente.")

Clave cargada correctamente.


In [14]:
url_alpha_vantage = "https://www.alphavantage.co/query"

parametros = {
    "function": "FX_DAILY",
    "from_symbol": "EUR",
    "to_symbol": "USD",
    "outputsize": "full",
    "apikey": api_key
}

respuesta = requests.get(
    url_alpha_vantage,
    params=parametros,
    timeout=30
)

respuesta.raise_for_status()

contenido = respuesta.json()

In [15]:
nombre_serie = "Time Series FX (Daily)"

if nombre_serie not in contenido:
    print("Respuesta recibida de Alpha Vantage:")
    print(contenido)

    raise ValueError(
        "Alpha Vantage no devolvió la serie diaria esperada."
    )

print("Datos descargados correctamente.")

Datos descargados correctamente.


In [16]:
datos_alpha = pd.DataFrame.from_dict(
    contenido[nombre_serie],
    orient="index"
)

datos_alpha.index = pd.to_datetime(
    datos_alpha.index
)

datos_alpha.index.name = "Date"

datos_alpha = datos_alpha.rename(columns={
    "1. open": "Open",
    "2. high": "High",
    "3. low": "Low",
    "4. close": "Close"
})

columnas_ohlc = [
    "Open",
    "High",
    "Low",
    "Close"
]

datos_alpha = datos_alpha[columnas_ohlc].astype(float)

datos_alpha = datos_alpha.sort_index()

print("Filas descargadas:", len(datos_alpha))
print("Primera fecha:", datos_alpha.index.min().date())
print("Última fecha:", datos_alpha.index.max().date())

display(datos_alpha.head())

Filas descargadas: 5000
Primera fecha: 2007-05-22
Última fecha: 2026-07-21


,Open,High,Low,Close
Date,,,,
2007-05-22,1.3467,1.3475,1.3438,1.3447
2007-05-23,1.3448,1.3501,1.3416,1.3460
2007-05-24,1.3458,1.3462,1.3415,1.3428
2007-05-25,1.3428,1.3472,1.3411,1.3442
2007-05-28,1.3449,1.3461,1.3445,1.3451


In [17]:
archivo_alpha = (
    ruta_proyecto
    / "data"
    / "raw"
    / "eurusd_alpha_vantage_diario.csv"
)

datos_alpha.to_csv(
    archivo_alpha,
    index=True
)

print("Archivo guardado en:")
print(archivo_alpha)

Archivo guardado en:
c:\TFM_EURUSD\data\raw\eurusd_alpha_vantage_diario.csv


In [18]:
datos_alpha = pd.read_csv(
    archivo_alpha,
    parse_dates=["Date"],
    index_col="Date"
)

In [19]:
comprobacion_ohlc_alpha = pd.DataFrame({
    "High_correcto": (
        datos_alpha["High"]
        >= datos_alpha[columnas_ohlc].max(axis=1)
    ),
    "Low_correcto": (
        datos_alpha["Low"]
        <= datos_alpha[columnas_ohlc].min(axis=1)
    )
})

print(
    "Filas con High correcto:",
    comprobacion_ohlc_alpha["High_correcto"].mean()
)

print(
    "Filas con Low correcto:",
    comprobacion_ohlc_alpha["Low_correcto"].mean()
)

Filas con High correcto: 1.0
Filas con Low correcto: 1.0


In [21]:
import numpy as np

rango_alpha = (
    datos_alpha["High"]
    - datos_alpha["Low"]
)

datos_alpha["Posicion_cierre"] = np.where(
    rango_alpha != 0,
    (
        datos_alpha["Close"]
        - datos_alpha["Low"]
    ) / rango_alpha,
    0.5
)

In [22]:
cierre_siguiente_alpha = (
    datos_alpha["Close"].shift(-1)
)

datos_alpha["Objetivo"] = np.where(
    cierre_siguiente_alpha.isna(),
    pd.NA,
    (
        cierre_siguiente_alpha
        > datos_alpha["Close"]
    ).astype(int)
)

datos_alpha["Objetivo"] = (
    datos_alpha["Objetivo"].astype("Int64")
)

In [23]:
alpha_hasta_validacion = datos_alpha.loc[
    datos_alpha.index <= "2024-12-31"
].copy()

alpha_hasta_validacion = (
    alpha_hasta_validacion
    .dropna(subset=["Objetivo"])
)

alpha_hasta_validacion["Objetivo"] = (
    alpha_hasta_validacion["Objetivo"].astype(int)
)

print(
    "Periodo analizado:",
    alpha_hasta_validacion.index.min().date(),
    "hasta",
    alpha_hasta_validacion.index.max().date()
)

print(
    "Filas disponibles:",
    len(alpha_hasta_validacion)
)

Periodo analizado: 2007-05-22 hasta 2024-12-31
Filas disponibles: 4595


In [24]:
pred_alpha = (
    alpha_hasta_validacion["Posicion_cierre"] < 0.5
).astype(int)

resultado_alpha = calcular_metricas(
    "Regla Posicion_cierre con Alpha Vantage",
    alpha_hasta_validacion["Objetivo"],
    pred_alpha
)

resultado_alpha

{'Modelo': 'Regla Posicion_cierre con Alpha Vantage',
 'Accuracy': 0.509031556039173,
 'Balanced_accuracy': 0.5090525448293501,
 'Precision': 0.5118421052631579,
 'Recall': 0.5051948051948052,
 'F1': 0.5084967320261438,
 'ROC_AUC': None}

In [25]:
alpha_entrenamiento = alpha_hasta_validacion.loc[
    alpha_hasta_validacion.index <= "2022-12-31"
].copy()

alpha_validacion = alpha_hasta_validacion.loc[
    (
        alpha_hasta_validacion.index
        > "2022-12-31"
    )
    & (
        alpha_hasta_validacion.index
        <= "2024-12-31"
    )
].copy()

pred_alpha_entrenamiento = (
    alpha_entrenamiento["Posicion_cierre"] < 0.5
).astype(int)

resultado_alpha_entrenamiento = calcular_metricas(
    "Alpha Vantage - entrenamiento",
    alpha_entrenamiento["Objetivo"],
    pred_alpha_entrenamiento
)

resultado_alpha_entrenamiento

{'Modelo': 'Alpha Vantage - entrenamiento',
 'Accuracy': 0.5099435305671495,
 'Balanced_accuracy': 0.5099875187664606,
 'Precision': 0.5143849206349206,
 'Recall': 0.5048685491723467,
 'F1': 0.5095823095823095,
 'ROC_AUC': None}

In [26]:
pred_alpha_validacion = (
    alpha_validacion["Posicion_cierre"] < 0.5
).astype(int)

resultado_alpha_validacion = calcular_metricas(
    "Alpha Vantage - validación",
    alpha_validacion["Objetivo"],
    pred_alpha_validacion
)

resultado_alpha_validacion

{'Modelo': 'Alpha Vantage - validación',
 'Accuracy': 0.5019157088122606,
 'Balanced_accuracy': 0.5020265507518797,
 'Precision': 0.49242424242424243,
 'Recall': 0.5078125,
 'F1': 0.5,
 'ROC_AUC': None}

In [27]:
alpha_hasta_validacion["Prediccion_simple"] = (
    alpha_hasta_validacion["Posicion_cierre"] < 0.5
).astype(int)

alpha_hasta_validacion["Acierto"] = (
    alpha_hasta_validacion["Prediccion_simple"]
    == alpha_hasta_validacion["Objetivo"]
).astype(int)

resultado_anual_alpha = (
    alpha_hasta_validacion
    .groupby(
        alpha_hasta_validacion.index.year
    )["Acierto"]
    .agg(["mean", "count"])
    .rename(columns={
        "mean": "Accuracy",
        "count": "Jornadas"
    })
    .round(4)
)

display(resultado_anual_alpha)

,Accuracy,Jornadas
Date,,
2007,0.5250,160
2008,0.4885,262
2009,0.5747,261
2010,0.4904,261
2011,0.4885,260
2012,0.4713,261
2013,0.5632,261
2014,0.5615,260
2015,0.5019,261


La regla basada únicamente en Posicion_cierre obtuvo aproximadamente un 50 % de accuracy con los datos de Alpha Vantage, tanto en entrenamiento como en validación. Además, su comportamiento anual se mantuvo alrededor del azar y no reprodujo el rendimiento cercano al 80 % observado con Yahoo Finance. Por esta razón, se considera que la relación detectada inicialmente no es suficientemente robusta y podría depender de particularidades de la fuente de datos utilizada.